# Chatterbox Nepali Fine-tuning (Clean venv)

Uses `uv` + isolated venv to avoid ALL Kaggle dependency conflicts.
**No session restart needed** — run all cells top to bottom.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
os.environ['TRAINING_ENV'] = 'kaggle'

REPO_URL = 'https://github.com/Firojpaudel/chatterbox-nepali.git'
REPO_DIR = '/kaggle/working/chatterbox-nepali'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)

subprocess.run(['pip', 'install', 'uv'], check=True)

VENV = '/kaggle/working/.venv'
VENV_PYTHON = f'{VENV}/bin/python'

if not os.path.exists(VENV):
    subprocess.run(['uv', 'venv', VENV, '--python', '3.12'], check=True)

def uv_install(*packages, index_url=None):
    cmd = ['uv', 'pip', 'install', '--python', VENV_PYTHON]
    if index_url:
        cmd += ['--index-url', index_url]
    cmd += list(packages)
    subprocess.run(cmd, check=True)

print('Installing PyTorch + CUDA...')
uv_install('torch==2.6.0', 'torchvision==0.21.0', 'torchaudio==2.6.0',
           index_url='https://download.pytorch.org/whl/cu124')

print('Installing chatterbox-tts...')
uv_install('-e', '.')

print('Installing training dependencies...')
uv_install('omegaconf', 'conformer', 's3tokenizer', 'safetensors',
           'pyloudnorm', 'wandb', 'huggingface-hub', 'librosa', 'soundfile')
uv_install('resemble-perth @ git+https://github.com/resemble-ai/Perth.git@master')

result = subprocess.run([VENV_PYTHON, '-c',
    'import numpy, torch; print(f"numpy={numpy.__version__}, torch={torch.__version__}, cuda={torch.cuda.is_available()}")'],
    capture_output=True, text=True)
print(f'Venv: {result.stdout.strip()}')
print('--- Setup Complete (no restart needed!) ---')

In [ ]:
# Cell 2: Download checkpoint
import os
from huggingface_hub import hf_hub_download
os.chdir('/kaggle/working/chatterbox-nepali')
ckpt_path = hf_hub_download(repo_id='officialuser/chatterbox-nepali',
    filename='t3_nepali_epoch_20.pt', token=os.environ.get('HF_TOKEN'))
dest = 't3_nepali_epoch_20.pt'
if os.path.exists(dest): os.remove(dest)
os.symlink(ckpt_path, dest)
print(f'Checkpoint ready at {dest}')

In [ ]:
# Cell 3: Download and extract dataset (pyarrow, no torchcodec)
import os
from pathlib import Path
from huggingface_hub import snapshot_download
import pyarrow.parquet as pq
from tqdm.auto import tqdm

dataset_repo = 'Firoj112/voxcpm-nepali-data'
local_dir = Path('finetuning_data')
wavs_dir = local_dir / 'wavs'
wavs_dir.mkdir(parents=True, exist_ok=True)

print(f'Downloading from {dataset_repo}...')
snapshot_download(repo_id=dataset_repo, repo_type='dataset', local_dir=local_dir,
    allow_patterns=['manifests/*.jsonl', 'data/*.parquet'],
    token=os.environ.get('HF_TOKEN'))

manifests = list((local_dir / 'manifests').glob('*.jsonl')) if (local_dir / 'manifests').exists() else []
print(f'Manifests: {[m.name for m in manifests]}')

parquet_files = sorted((local_dir / 'data').glob('*.parquet'))
print(f'Found {len(parquet_files)} parquet files')
for pf in parquet_files:
    table = pq.read_table(pf, columns=['audio'])
    print(f'{pf.name}: {len(table)} rows')
    for i in tqdm(range(len(table)), desc=pf.name):
        audio_struct = table['audio'][i].as_py()
        if not audio_struct: continue
        audio_bytes = audio_struct.get('bytes')
        path = audio_struct.get('path', f'audio_{i}.wav')
        if not audio_bytes: continue
        save_path = wavs_dir / Path(path).name
        if not save_path.exists():
            with open(save_path, 'wb') as f:
                f.write(audio_bytes)

wav_count = len(list(wavs_dir.glob('*.wav')))
print(f'Dataset ready: {wav_count} WAV files')

In [ ]:
# Cell 4: Training (clean venv + fp16 + reduced batch to fit T4 14.5GB)
import os
os.chdir('/kaggle/working/chatterbox-nepali')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

VENV_BIN = '/kaggle/working/.venv/bin'

# batch_size=4 x accum_steps=4 x 2 GPUs = effective batch 32
# --fp16 enables mixed precision (halves memory usage)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  {VENV_BIN}/torchrun --nproc_per_node=2 src/chatterbox/train_nepali.py \
    --manifest finetuning_data/manifests/train_manifest.jsonl \
    --wav_dir finetuning_data/wavs \
    --resume_t3_weights t3_nepali_epoch_20.pt \
    --batch_size 4 \
    --accum_steps 4 \
    --epochs 50 \
    --lr 2e-5 \
    --save_every 5 \
    --num_workers 2 \
    --distributed \
    --fp16 \
    --push_to_hub officialuser/chatterbox-nepali

In [ ]:
# Cell 5: Push final model to HF
from huggingface_hub import HfApi
import os, glob

os.chdir('/kaggle/working/chatterbox-nepali')
api = HfApi()
repo_id = 'officialuser/chatterbox-nepali'
token = os.environ.get('HF_TOKEN')
final_model = 't3_mtl_nepali_final.safetensors'

if not token:
    print('HF_TOKEN not set.')
elif os.path.exists(final_model):
    print(f'Pushing {final_model}...')
    api.upload_file(path_or_fileobj=final_model, path_in_repo=final_model, repo_id=repo_id, repo_type='model', token=token)
    print('Done!')
else:
    checkpoints = sorted(glob.glob('t3_nepali_epoch_*.pt'))
    if checkpoints:
        latest = checkpoints[-1]
        print(f'Pushing latest checkpoint: {latest}')
        api.upload_file(path_or_fileobj=latest, path_in_repo=latest, repo_id=repo_id, repo_type='model', token=token)
        print('Done!')
    else:
        print('No model or checkpoints found.')